# 2.4 Maxwell's Equations

[Peter Monk: "Finite Element Methods for Maxwell's Equations"]

## Magnetostatic field generated by a permanent magnet

With the magnetic flux density $B$, the magnetic field $H$, and a given magnetization $M$, the equations of magnetostatics are

$$
\DeclareMathOperator{\Grad}{grad}
\DeclareMathOperator{\Curl}{curl}
\DeclareMathOperator{\Div}{div}
B = \mu (H + M), \quad \Div B = 0, \quad \Curl H = 0
$$

Introducing a vector potential $A$ such that $B = \Curl A$, and combining the equations, we get

$$
\Curl \mu^{-1} \Curl A = \Curl M
$$

In weak form: Find $A \in H(\Curl)$ such that

$$
\int \mu^{-1} \Curl A \Curl v = \int M \Curl v \qquad  \forall \, v \in H(\Curl)
$$

Usually, the permeability $\mu$ is given as
$\mu = \mu_r \mu_0$, with $\mu_0 = 4 \pi 10^{-7}$ the permeability of vacuum.

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

Geometric model and meshing of a bar magnet. The magnet is cut out of the air box, and `Glue` connects both solids to one geometry with a conforming interface:

In [ ]:
box = Box((-3,-3,-3), (3,3,3))
box.faces.name = "outer"

magnet = Cylinder((-1,0,0), X, r=0.3, h=2)
magnet.solids.name = "magnet"
magnet.faces.col = (1,0,0)

air = box - magnet
air.solids.name = "air"

shape = Glue([air, magnet])
Draw (shape, clipping={ "z" : -1, "function" : True })

mesh = shape.GenerateMesh(maxh=2, curvaturesafety=1).Curve(3)

In [ ]:
mesh.GetMaterials(), mesh.GetBoundaries()

Define the finite element space and the variational equation.

* The vector potential is not unique, we may add an arbitrary gradient field. A cheap possibility for gauging is to remove all high order basis functions which are gradients (flag `nograds`).
* The gradients within the lowest order edge element space are still there. To obtain a regular system matrix, we regularize by adding a very small $L_2$ term.

In [ ]:
fes = HCurl(mesh, order=3, nograds=True)
print ("ndof =", fes.ndof)
u,v = fes.TnT()

mu0 = 4*pi*1e-7
mur = mesh.MaterialCF({"magnet" : 1000}, default=1)
nu = 1/(mu0*mur)
mag = mesh.MaterialCF({"magnet" : (1,0,0)}, default=(0,0,0))

lhs = nu*curl(u)*curl(v)*dx + 1e-8*nu*u*v*dx
rhs = mag*curl(v) * dx("magnet")

We solve by a conjugate gradient iteration with a BDDC preconditioner. The essential boundary condition $A \times n = 0$ on the outer boundary is given as an argument. `Solve` assembles the system, sets up the preconditioner, and solves, all using task-parallelization:

In [ ]:
gfu = Solve(lhs==rhs, u[BND("outer")]==(0,0,0),
            preconditioners.BDDC(),
            solvers.CGSolver.Creator(printrates=True, tol=1e-8))

The magnetic flux density is $B = \Curl A$:

In [ ]:
Draw (curl(gfu), mesh, "B-field", draw_surf=False, \
      clipping = { "z" : -1, "function":True}, \
      vectors = { "grid_size":50}, min=0, max=2e-5);